# Phase A — Detection MVP on Climbing Holds Dataset

Trains `v9-t` and `v9-s` (detection only) on the 19-image hand-labelled climbing dataset.

**Prereqs:**
1. Drive folder `My Drive/climbing-holds/` exists with `data/climbing_holds.zip` uploaded
2. Fork at `https://github.com/ob-choco/YOLO` has the `feature/climbing-seg` branch pushed

**Output:** checkpoints + visualizations to `My Drive/climbing-holds/runs/phaseA-<size>-<datetime>/`

**Runtime:** Colab Free T4 → ~1–3 hours per model size.

**Spec / plan refs:**
- `docs/superpowers/specs/2026-05-06-climbing-holds-segmentation-design.md`
- `docs/superpowers/plans/2026-05-06-climbing-holds-segmentation.md` Tasks 7, 8

## 1. Mount Drive and clone the fork

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, subprocess
os.chdir("/content")
if not os.path.exists("YOLO"):
    subprocess.check_call([
        "git", "clone", "-b", "feature/climbing-seg",
        "https://github.com/ob-choco/YOLO.git",
    ])
os.chdir("/content/YOLO")
subprocess.check_call(["git", "pull"])
print(subprocess.check_output(["git", "log", "--oneline", "-3"]).decode())

## 2. Install dependencies

Colab pre-installs torch with CUDA; the rest of `requirements.txt` covers Lightning/Hydra/PyCOCO/etc.

In [ ]:
# Important: don't reinstall torch / torchvision — Colab ships them with CUDA
# baked in, and pip would otherwise replace the wheel with a CPU-only one. We
# strip those lines from requirements.txt before invoking pip.
# (Also skip numpy<2 pin: Colab's torch 2.10+ is fine with numpy 2.x.)
!grep -vE "^(torch|torchvision)$" requirements.txt > /tmp/req-no-torch.txt
!pip install -q -r /tmp/req-no-torch.txt
!pip install -q pycocotools

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("\n❌ NO GPU DETECTED.")
    print("   1) Check Colab menu: Runtime → Change runtime type → T4 GPU")
    print("   2) Re-run cells 1, 2, 3 after switching")
    print("   Training cells below will fail with 'No supported gpu backend found!'")
    raise SystemExit("GPU runtime required — abort here so the failure is obvious.")

## 3. Unzip dataset from Drive

Skip if already unzipped (Colab session was reused).

In [ ]:
import os, subprocess
data_root = "/content/YOLO/data/climbing_holds"
if not os.path.exists(f"{data_root}/annotations/instances_train.json"):
    !mkdir -p /content/YOLO/data
    !unzip -q -o /content/drive/MyDrive/climbing-holds/data/climbing_holds.zip -d /content/YOLO/
n_train = int(subprocess.check_output(f"ls {data_root}/images/train 2>/dev/null | wc -l", shell=True))
n_val = int(subprocess.check_output(f"ls {data_root}/images/val 2>/dev/null | wc -l", shell=True))
# Slim zip ships only the 15 labeled train images + 4 labeled val images.
# (Earlier instructions had a fat zip with all 2273 images — slim is functionally
# identical for training because the data loader filters by COCO annotation.)
print(f"images/train: {n_train} files (expect 15 for slim, 1061 for fat)")
print(f"images/val:   {n_val} files (expect 4 for slim, 1212 for fat)")
!ls {data_root}/annotations/   # expect instances_train.json, instances_val.json
sample = subprocess.check_output(f"ls {data_root}/images/train | head -1", shell=True).decode().strip()
if sample:
    full = f"{data_root}/images/train/{sample}"
    size = os.path.getsize(full) if os.path.exists(full) else 0
    print(f"\nFirst image: {sample} — size {size} bytes")
    if size == 0:
        print("\n❌ FIRST IMAGE IS EMPTY OR A BROKEN SYMLINK.")
        print("   Recreate the zip on the Mac with real file contents (not symlinks).")
        raise SystemExit("Dataset zip is broken — see message above.")
if n_train == 0 or n_val == 0:
    raise SystemExit(f"Dataset extraction yielded 0 files. Re-check the zip on Drive.")

## 4. Download pretrained weights (v9-t and v9-s detection)

From the upstream MultimediaTechLab/YOLO release. Cached on Drive after the first run.

In [ ]:
import os, urllib.request, shutil
os.makedirs("weights", exist_ok=True)
drive_pretrained = "/content/drive/MyDrive/climbing-holds/pretrained"
os.makedirs(drive_pretrained, exist_ok=True)

# v1.0-alpha release ships .pt files (not .ckpt). v9-t.pt = 12.2MB, v9-s.pt = 39.9MB
for ckpt in ["v9-t.pt", "v9-s.pt"]:
    drive_path = os.path.join(drive_pretrained, ckpt)
    local_path = os.path.join("weights", ckpt)
    if os.path.exists(drive_path):
        shutil.copy(drive_path, local_path)
        print(f"Loaded {ckpt} from Drive cache ({os.path.getsize(local_path)} bytes)")
        continue
    url = f"https://github.com/MultimediaTechLab/YOLO/releases/download/v1.0-alpha/{ckpt}"
    try:
        urllib.request.urlretrieve(url, local_path)
        shutil.copy(local_path, drive_path)
        print(f"Downloaded {ckpt} ({os.path.getsize(local_path)} bytes), cached to Drive")
    except Exception as e:
        print(f"⚠️  Failed to download {ckpt}: {e}")
        print(f"    Training will fall back to weight=False (random init) which on 15 images won't")
        print(f"    produce visible detections in 50 epochs. Investigate the URL before continuing.")

## 5a. Train v9-t (detection)

50 epochs, batch=8, image_size=640, T4 GPU.

Override flags learned during the laptop smoke test:
- `accelerator=gpu device=1` (Colab T4)
- `image_size=[640,640]` (list, not int)
- `use_wandb=False` (no API key on Colab)

In [ ]:
import datetime, os, subprocess
run_name = f"phaseA-v9-t-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
weight_arg = "weights/v9-t.pt" if os.path.exists("weights/v9-t.pt") else "False"
if weight_arg == "False":
    print("⚠️  weights/v9-t.pt not present — training from random init.")
# 2000 epochs for the random-init class head to learn — 80→2 class shape
# mismatch on the pretrained ckpt forces the class_conv layers to be re-initialized,
# and on 15 training images they need many more passes to differentiate hold vs
# volume vs background. T4 cost ≈ 30-60 min for v9-t at this size.
cmd = (
    "python yolo/lazy.py task=train model=v9-t dataset=climbing_holds "
    "dataset.path=/content/YOLO/data/climbing_holds "
    "task.data.batch_size=8 image_size=[640,640] task.epoch=2000 "
    "accelerator=gpu device=1 use_wandb=False "
    f"weight={weight_arg} name={run_name}"
)
print("$", cmd)
result = subprocess.run(cmd + " 2>&1", shell=True, capture_output=True, text=True, bufsize=1)
print(result.stdout or "(no stdout captured)")
if result.returncode != 0:
    raise RuntimeError(f"v9-t training failed with exit code {result.returncode}.")
print("v9-t run:", run_name)

## 5b. Train v9-s (detection)

In [ ]:
run_name_s = f"phaseA-v9-s-{datetime.datetime.now().strftime('%Y%m%d-%H%M')}"
weight_arg_s = "weights/v9-s.pt" if os.path.exists("weights/v9-s.pt") else "False"
if weight_arg_s == "False":
    print("⚠️  weights/v9-s.pt not present — training from random init.")
cmd_s = (
    "python yolo/lazy.py task=train model=v9-s dataset=climbing_holds "
    "dataset.path=/content/YOLO/data/climbing_holds "
    "task.data.batch_size=8 image_size=[640,640] task.epoch=2000 "
    "accelerator=gpu device=1 use_wandb=False "
    f"weight={weight_arg_s} name={run_name_s}"
)
print("$", cmd_s)
result = subprocess.run(cmd_s + " 2>&1", shell=True, capture_output=True, text=True, bufsize=1)
print(result.stdout or "(no stdout captured)")
if result.returncode != 0:
    raise RuntimeError(f"v9-s training failed with exit code {result.returncode}.")
print("v9-s run:", run_name_s)

## 6. Save artifacts to Drive

Each run directory under `runs/train/<name>/` contains: `last.ckpt`, `best.ckpt` (if checkpoints fired), `val_pred_epoch*.png`, hydra logs.

In [ ]:
import shutil
for run in (run_name, run_name_s):
    src = f"/content/YOLO/runs/train/{run}"
    dst = f"/content/drive/MyDrive/climbing-holds/runs/{run}"
    if os.path.exists(src):
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Saved:", dst)
    else:
        print("Run dir missing (training failed?):", src)

## 7. Quick visual / metrics review

Verify val PNG overlays look sensible before moving to Phase B.

In [ ]:
from IPython.display import Image as _Img, display
import glob, pandas as pd
for run in (run_name, run_name_s):
    pngs = sorted(glob.glob(f"runs/train/{run}/val_pred_epoch*.png"))
    if pngs:
        print(f"\n=== {run} (latest val PNG) ===")
        display(_Img(pngs[-1]))
    csvs = glob.glob(f"runs/train/{run}/**/metrics.csv", recursive=True)
    if csvs:
        print("metrics tail:", pd.read_csv(csvs[0]).tail(5))

## 8. Diagnostic — class-head weight statistics + threshold sweep

If cell 7's val PNG is empty (no boxes) or saturated with low-confidence noise,
this cell helps tell whether the model is genuinely under-trained vs.
something else is broken:

- **`class_conv` weight statistics** — small `std` (≪ 0.01) on the matched-shape
  class layers means the random-init class head barely moved during training;
  raise epoch count or check learning rate.
- **NMS threshold sweep** — re-render predictions at 0.5 / 0.1 / 0.01. If higher
  thresholds yield zero boxes but lower thresholds yield many, the model has
  learned something but with low confidence — likely needs more epochs.

In [ ]:
import torch, glob, os, subprocess
from IPython.display import Image as _Img, display

for run in (run_name, run_name_s):
    print(f"\n=========== {run} ===========")
    ckpts = sorted(glob.glob(f"runs/train/{run}/checkpoints/*.ckpt"))
    if not ckpts:
        print("  No checkpoint produced — training likely failed early.")
        continue
    ckpt_path = ckpts[-1]
    print(f"  ckpt: {ckpt_path}")

    # 1) class_conv weight stats — if std is near zero, the head barely trained.
    state = torch.load(ckpt_path, map_location="cpu", weights_only=False).get("state_dict", {})
    print("\n  class_conv weight stats (lower std = head not yet learning):")
    for k in sorted(state.keys()):
        if "class_conv" in k and ("weight" in k and ".bias" not in k):
            w = state[k].float()
            print(f"    {k:60s} shape={tuple(w.shape)} std={w.std():.4f} max_abs={w.abs().max():.4f}")

    # 2) Threshold sweep: re-run validation with various NMS confidence thresholds
    model = "v9-t" if "v9-t" in run else "v9-s"
    print("\n  Validation @ different NMS confidence thresholds:")
    for thresh in [0.5, 0.1, 0.01]:
        diag_name = f"diag_{run}_t{int(thresh*1000):04d}"
        cmd = (
            f"python yolo/lazy.py task=validation model={model} dataset=climbing_holds "
            f"dataset.path=/content/YOLO/data/climbing_holds "
            f"image_size=[640,640] task.data.batch_size=2 "
            f"accelerator=gpu device=1 use_wandb=False "
            f"weight={ckpt_path} task.nms.min_confidence={thresh} "
            f"name={diag_name}"
        )
        result = subprocess.run(cmd + " 2>&1", shell=True, capture_output=True, text=True)
        ap_lines = [
            l.strip() for l in result.stdout.splitlines()
            if "AP @     .5" in l or "AP @ .5:.95" in l or "Recorded" in l
        ]
        print(f"    threshold {thresh}:")
        for l in ap_lines[:3]:
            print(f"      {l}")
        png = sorted(glob.glob(f"runs/validation/{diag_name}/val_pred_epoch*.png"))
        if png:
            print(f"      val PNG: {png[-1]}")
            display(_Img(png[-1]))